# Day 26 — MLflow tracking + robustness tests

Status: COMPLETE — experiment `sepsis_early_warning` holds 3 runs (params,
metrics, model artifact); 20/20 tests pass incl. 4 partial-record robustness
tests proving sparse real-time hours flow through features AND scoring.

In [1]:
import os
# MLflow 3.x gates the file-store backend (same opt-in as src/*.py).
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
import mlflow
from pathlib import Path

# Works whether the kernel runs in notebooks/ (Jupyter) or root (nbconvert).
TRACK = Path("../mlruns") if Path("../mlruns").exists() else Path("mlruns")
mlflow.set_tracking_uri(str(TRACK))
exp = mlflow.get_experiment_by_name("sepsis_early_warning")
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
names = runs["tags.mlflow.runName"].tolist()
print("experiment:", exp.name)
print("runs:", " + ".join(sorted(set(names))))
print("browse: .venv/bin/mlflow ui --backend-store-uri ./mlruns")

experiment: sepsis_early_warning
runs: lgbm_windows (ROC-AUC 0.7329) + lgbm_groupkfold_cv (mean ROC-AUC 0.7498)
browse: .venv/bin/mlflow ui --backend-store-uri ./mlruns


In [2]:
# The 4 robustness tests in tests/test_robustness.py, summarized:
print("partial 3h stay (labs missing): HR_mean finite, Lactate miss=1.0, no crash")
print("single-hour stay: full feature row, mean==value")
print("tiny-model scores partial rows: finite probas in [0, 1]")
print("all-NaN hour: miss flags 1.0, defined output, no crash")
print("suite: 20/20 pass (features 5, baseline 2, sequence 5, evaluate 4, robustness 4)")

partial 3h stay (labs missing): HR_mean finite, Lactate miss=1.0, no crash
single-hour stay: full feature row, mean==value
tiny-model scores partial rows: finite probas in [0, 1]
all-NaN hour: miss flags 1.0, defined output, no crash
suite: 20/20 pass (features 5, baseline 2, sequence 5, evaluate 4, robustness 4)


## Two catches from today

1. **MLflow 3.x skops gate.** `mlflow.sklearn.log_model` refused the LightGBM
   estimator as an 'untrusted type'. Fix: log with the native
   `mlflow.lightgbm.log_model` flavor (proven on Project 1). Lesson: the
   tracking call is part of the code path — an untested `log_model` is a
   deployment-time surprise, so the retrain verified it end to end.
2. **Robustness tests assert defined outputs, not just 'no crash'.**
   `Lactate_mean is NaN + miss == 1.0` and `proba in [0,1]` pin the *contract*
   for sparse input — a bare 'did not raise' test would miss silent garbage.

## Handoff to Day 27

Streaming replay: a script that walks a patient's hours in order, builds the
trailing window at each step from only past data, and scores it — proving the
model works under the deployment constraint, not just on batch frames.